# FI-Ran clear-cut: diagnosis and fix of three convergence issues

**Meeting summary -- branch `debug/tl-convergence-isolation`**

**TL;DR:** In the FI-Ran clear-cut simulation (Ranskalankorpi), three nested
iterative solves did not always converge within their iteration budget: the canopy's outer
Picard loop (`mlm_canopy`), the wet-leaf energy balance in rainfall interception
(`interception`), and the dry-leaf gas exchange loop (`planttype`). Relaxation (gamma
damping) was added to all three, including a `prev_err`/`gamma` bugfix in `interception.py`
that had left its oscillation damping dead code. A direct, same-window baseline-vs-fixed
comparison over the full 1.5.-30.9.2022 season (7297 timesteps, same forcing) shows the
outer loop's non-convergent timesteps drop from **1418/7297 (19.4%)** to **1/7297 (0.01%)**,
`interception` itermax notices from **40** to **0**, and `planttype` itermax/unrealistic
notices from **4922** to **5**.


## Background: what was broken

`CanopyModel.run()` (`pyAPES/canopy/mlm_canopy.py`) solves the canopy's temperature, H2O
and CO2 profiles iteratively (Picard loop) at every timestep, and two other modules solve
their own energy balance on their own nested loop inside that:

- `Interception.run()` -- wet-leaf temperature (rainfall evaporation)
- `PlantType.leaf_gas_exchange()` -- dry-leaf temperature (photosynthesis + transpiration),
  solved separately for each of 3 planttypes (spruce/decid/shrubs) and both leaf types
  (sunlit/shaded)

When any of these failed to converge within its iteration budget, the model either
continued with an inexact value ("tolerable") or fell back to a coarser well-mixed
assumption ("switched to WMA"). The diagnosis started from a longer historical log
(`ran_22_k7.log`, not preserved on disk -- see caveats below) that showed this happening
often. To get a precise, reproducible before/after number, the cells below instead run
branch `case_ranskalankorpi` (this branch's pre-fix ancestor, own parameters untouched) over
the *exact same* 1.5.-30.9.2022 window and forcing as the fixed run later in this notebook:


In [ ]:
import os
import sys

assert os.path.basename(os.getcwd()) == 'debug', (
    f"expected to run with cwd=debug/ (this notebook's own directory), got {os.getcwd()!r} -- adjust paths below if not")
sys.path.insert(0, os.path.abspath('..'))

import re
import pickle
from collections import Counter
import numpy as np
import pandas as pd
import xarray as xr
%matplotlib inline
import matplotlib.pyplot as plt

COLOR_CRITICAL = '#d03b3b'   # validated status palette (dataviz skill)
COLOR_GOOD = '#0ca30c'
MODULE_ORDER = ['mlm_canopy', 'interception', 'planttype']
MODULE_LABELS = {
    'mlm_canopy': 'mlm_canopy\n(outer Picard loop)',
    'interception': 'interception\n(wet-leaf loop)',
    'planttype': 'planttype\n(leaf_gas_exchange)',
}
MODULE_MAP = {
    'mlm_canopy': 'canopy.mlm_canopy',
    'interception': 'canopy.interception',
    'planttype': 'planttype.planttype',
}

N_TIMESTEPS = 7297  # 1.5.-30.9.2022, 30-min dt -- both baseline and fixed runs


def count_debug_messages(log_path, module_map):
    counts = Counter()
    pattern = re.compile(r'DEBUG (pyAPES\.\S+) ')
    with open(log_path) as f:
        for line in f:
            m = pattern.match(line)
            if not m:
                continue
            for key, needle in module_map.items():
                if needle in m.group(1):
                    counts[key] += 1
    return counts


def count_outer_loop_outcome(log_path, n_timesteps):
    """mlm_canopy outer-loop outcome per timestep. A nan-blowup timestep also logs
    'Switched to WMA assumption' right after (see mlm_canopy.py's fall-through from
    the nan-check into the WMA-switch branch), so it's already included in `switched`
    -- reported separately here only as a diagnostic, not double-counted."""
    tolerable = 0
    switched = 0
    blowup = 0
    with open(log_path) as f:
        for line in f:
            if 'canopy.mlm_canopy' not in line:
                continue
            if 'Maximum iterations reached but error tolerable' in line:
                tolerable += 1
            elif 'Switched to WMA assumption' in line:
                switched += 1
            elif 'Solution of profiles blowing up' in line:
                blowup += 1
    converged = n_timesteps - tolerable - switched
    return {'converged': converged, 'tolerable': tolerable, 'switched_to_wma': switched,
            'switched_to_wma_from_nan_blowup': blowup}


try:
    baseline_counts = count_debug_messages('../logs/ran_22_baseline.log', MODULE_MAP)
    baseline_outcome = count_outer_loop_outcome('../logs/ran_22_baseline.log', N_TIMESTEPS)
except FileNotFoundError:
    # logs/ is gitignored -- reproduce with debug/run_baseline_ran22_case_ranskalankorpi.py
    # (see its docstring for the git-worktree recipe) if this file is missing.
    baseline_counts = Counter({'mlm_canopy': 1422, 'interception': 40, 'planttype': 4922})
    baseline_outcome = {'converged': 5879, 'tolerable': 186, 'switched_to_wma': 1232,
                         'switched_to_wma_from_nan_blowup': 4}

print(f'Baseline (case_ranskalankorpi, pre-fix, {N_TIMESTEPS} timesteps):')
for m in MODULE_ORDER:
    print(f'  {m}: {baseline_counts[m]} DEBUG messages')
print()
print('Outer-loop (mlm_canopy) outcome:')
print(' ', baseline_outcome)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
values = [baseline_counts[m] for m in MODULE_ORDER]
bars = ax.bar([MODULE_LABELS[m] for m in MODULE_ORDER], values, color=COLOR_CRITICAL, width=0.6)
for bar, v in zip(bars, values):
    ax.annotate(str(v), (bar.get_x() + bar.get_width() / 2, v), ha='center', va='bottom', fontsize=11)
ax.set_ylabel('convergence DEBUG messages (count)')
ax.set_title(f'Baseline (case_ranskalankorpi, pre-fix): non-convergence by module, {N_TIMESTEPS} timesteps')
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()


## Three problems found and their fixes

| # | module | problem | fix |
|---|---|---|---|
| 1 | `mlm_canopy.run()` (`pyAPES/canopy/mlm_canopy.py`) | the outer loop's relaxation factor `gam` couldn't drop low enough in oscillating cases | lowered `gam_floor` from `0.25` to `0.01` |
| 2 | `Interception.run()` (`pyAPES/canopy/interception.py`) | wet-leaf temperature loop had no relaxation at all (plain fixed-point) | added oscillation-adaptive gamma relaxation (`gamma=0.75` start, `gamma_floor=0.01`) |
| 3 | `PlantType.leaf_gas_exchange()` (`pyAPES/planttype/planttype.py`) | dry-leaf temperature loop had no relaxation at all, same issue as interception | added the same oscillation-adaptive relaxation (`gamma=1.0` start, `gamma_floor=0.05`) |

Each fix was first validated in a standalone sandbox notebook (a copy of just the loop,
run against already-captured real forcing data, so candidate values could be tried without
re-running the full source code each time):

- `debug_mlm_canopy_convergence.ipynb` -- 19 captured non-convergent timesteps,
  `gam0=0.1, gam_floor=0.15`: **18/19 converge** (vs. the `gam_floor=0.01` that ended up in source)
- `debug_interception_relaxation.ipynb` -- 1392 timesteps (June 2022):
  baseline (no relaxation) **3/1392 fail**; `gamma=0.5`: **0/1392 fail**
  (but mean iteration count roughly doubles across the whole dataset)
- `debug_planttype_leaf_temperature_relaxation.ipynb` -- 8352 combinations
  (1392 timesteps x 3 planttypes x 2 leaf types): baseline **11/8352 fail**
  (all `spruce`/`sunlit`, midday hours); oscillation-adaptive relaxation:
  **0/8352 fail**, mean iteration count **essentially unchanged** (3.77 -> 3.76)

**Two implementation bugs were also found and fixed along the way** that would otherwise
have prevented the fixes from working at all: in `interception.py` the relaxation formula
referenced a mismatched array shape (`ValueError: shapes (7,) (41,)`), and in `planttype.py`
the oscillation branch called `np.max(a, b)` instead of `max(a, b)` (`TypeError`, crashed
the full run about 80% through the June simulation).

## Result: full 1.5.-30.9.2022 run (5 months, 7297 timesteps) with all three fixes

The run below covers the FI-Ran default simulation window
(`mlm_parameters_FI_Ran.gpara`, 1.5.-30.9.2022, **7297 timesteps**, 30-min timestep) --
the exact same window and forcing as the `case_ranskalankorpi` baseline above -- executed
against the current, fixed source code (`PYTHONPATH=. python3 debug/run_ran22_full_season.py`,
~342 s, `results/ran_22_fixed.nc` / `logs/ran_22_fixed.log`). Not simulated, a real run,
and now a genuine same-window, same-forcing before/after (the baseline above is too).


In [ ]:
current_counts = count_debug_messages('../logs/ran_22_fixed.log', MODULE_MAP)
current_outcome = count_outer_loop_outcome('../logs/ran_22_fixed.log', N_TIMESTEPS)

print(f'Current run (1.5.-30.9.2022, all 3 fixes, {N_TIMESTEPS} timesteps):')
for m in MODULE_ORDER:
    print(f'  {m}: {current_counts[m]} DEBUG messages')
print()
print('Outer-loop (mlm_canopy) outcome:')
print(' ', current_outcome)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# --- left: per-module debug-message count, baseline (case_ranskalankorpi) vs now (fixed) ---
ax = axes[0]
x = np.arange(len(MODULE_ORDER))
width = 0.35
before_vals = [baseline_counts[m] for m in MODULE_ORDER]
after_vals = [current_counts[m] for m in MODULE_ORDER]
ax.bar(x - width / 2, before_vals, width, color=COLOR_CRITICAL, label='before (case_ranskalankorpi)')
ax.bar(x + width / 2, after_vals, width, color=COLOR_GOOD, label='now (fixed)')
for xi, v in zip(x - width / 2, before_vals):
    ax.annotate(str(v), (xi, v), ha='center', va='bottom', fontsize=9)
for xi, v in zip(x + width / 2, after_vals):
    ax.annotate(str(v), (xi, v), ha='center', va='bottom', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels([MODULE_LABELS[m] for m in MODULE_ORDER])
ax.set_ylabel('convergence DEBUG messages (count)')
ax.set_title(f'DEBUG messages per module: before vs now (same {N_TIMESTEPS}-timestep window)')
ax.legend(fontsize=8)
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)

# --- right: outer-loop outcome, same window/forcing both sides, so raw counts are
#     directly comparable (unlike an earlier version of this notebook, which compared
#     a June-only fixed run against a longer, differently-sized historical log) ---
ax = axes[1]
outcome_labels = ['converged', 'tolerable', 'switched_to_wma']
before_v = [baseline_outcome[k] for k in outcome_labels]
after_v = [current_outcome[k] for k in outcome_labels]
x2 = np.arange(len(outcome_labels))
ax.bar(x2 - width / 2, before_v, width, color=COLOR_CRITICAL, label='before (case_ranskalankorpi)')
ax.bar(x2 + width / 2, after_v, width, color=COLOR_GOOD, label='now (fixed)')
for xi, v in zip(x2 - width / 2, before_v):
    ax.annotate(str(v), (xi, v), ha='center', va='bottom', fontsize=9)
for xi, v in zip(x2 + width / 2, after_v):
    ax.annotate(str(v), (xi, v), ha='center', va='bottom', fontsize=9)
ax.set_xticks(x2)
ax.set_xticklabels(outcome_labels, rotation=15)
ax.set_ylabel(f'timesteps (of {N_TIMESTEPS} total)')
ax.set_title('mlm_canopy outer-loop outcome: before vs now')
ax.legend(fontsize=8)
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()


## Notes and caveats

- **Genuine same-window comparison**: both runs use the identical 1.5.-30.9.2022 forcing
  and 7297 timesteps -- the baseline is branch `case_ranskalankorpi` (own parameters
  untouched, i.e. before both the param retuning and the convergence fixes that followed
  it on this branch) with only `gpara['start_time']`/`['end_time']` overridden to match.
  Reproduce with `debug/run_baseline_ran22_case_ranskalankorpi.py` (see its docstring for
  the git-worktree recipe) and `debug/run_ran22_full_season.py`.
- The outer loop's non-convergent timesteps drop from **1418/7297 (19.4%)** to **1/7297
  (0.014%)**. 4 of the baseline's 1232 "switched to WMA" timesteps were preceded by the
  Picard loop's T/H2O/CO2 profiles actually blowing up to `nan` (`case_ranskalankorpi`
  lacks the outer loop's `gam_floor=0.01`, so oscillations there are not damped enough to
  prevent this) -- none of that happens in the fixed run.
- `interception` itermax notices drop from **40** to **0**, confirming both the relaxation
  and the `gamma`/`prev_err` bugfix (an earlier, June-only validation on this branch still
  had 7 residual `interception` notices caused by that bug, before it was found and fixed).
- `planttype` itermax/unrealistic-temperature notices drop from **4922** to **5** -- the
  baseline's `planttype` count is the largest of the three because it logs once per
  planttype (spruce/decid/shrubs) x leaf type (sunlit/shaded) x timestep, i.e. up to 6
  messages for a single problematic timestep.
- The original diagnosis (see TL;DR/Background) started from a longer historical log
  (`ran_22_k7.log`) that isn't preserved on disk (`logs/` is gitignored); the
  `case_ranskalankorpi` baseline above supersedes it with a precise, reproducible number
  over an identical window.
- This branch (`debug/tl-convergence-isolation`) is a research branch. The debug-capture
  instrumentation (`pyAPES/utils/debug_capture.py`, hooks in `mlm_canopy.py`) is opt-in and
  has no effect on normal runs, but it and this notebook and the other
  `debug/debug_*` files should be cleaned up or dropped before any merge to `main`.

## Next steps

1. A light code review of the three changes (`mlm_canopy.py`, `interception.py`,
   `planttype.py`) before considering them for further use.
2. ~~A longer validation run (e.g. a full growing season or multiple years) to confirm the
   result holds outside of June as well.~~ **Done** -- see the full-season, same-window
   baseline-vs-fixed result above. A multi-year run (2023-2025, also in the forcing file)
   would extend this further.
3. Clean up / remove the debug instrumentation (`debug_capture.py`) and research notebooks
   before merging this branch into `main`, if it gets merged.


## Extended analysis: did the fix change the results, not just convergence?

Fixing non-convergence changes *how* a timestep's profiles are solved, so it's worth
checking whether it also changed *what* they converge to. Below: canopy air/leaf profiles
(`Tleaf`, `H2O`, `CO2`) averaged over the full 1.5.-30.9.2022 run and split into daytime
(08:00-16:00) and nighttime (22:00-06:00), plus canopy-level flux time series (`NEE`, `GPP`,
`Reco`, `H`, `LE`, `G`, `Rnet`), both baseline (`case_ranskalankorpi`) vs now (fixed) --
same run outputs as the convergence counts above (`results/ran_22_baseline.nc` /
`results/ran_22_fixed.nc`).

- `Tleaf` = `canopy_Tleaf`: the model's own LAI- and sunlit/shaded-fraction-weighted mean
  leaf temperature per layer, already averaged across all 3 planttypes and both leaf types
  inside `CanopyModel.run()` -- exactly the weighting the fix targets, so used directly
  rather than recomputed here.
- `H2O`/`CO2` = `canopy_h2o`/`canopy_co2`: the canopy air-space scalar profiles (the other
  two quantities the outer Picard loop solves for, alongside `T`/`Tleaf`).
- `G` (ground heat flux) has no canopy-level equivalent -- `ffloor_ground_heat` (forest
  floor) is used instead.


In [ ]:
baseline_ds = xr.open_dataset('../results/ran_22_baseline.nc').squeeze('simulation', drop=True)
fixed_ds = xr.open_dataset('../results/ran_22_fixed.nc').squeeze('simulation', drop=True)

assert len(baseline_ds['date']) == len(fixed_ds['date']), 'baseline and fixed runs must cover the same window'

z = fixed_ds['canopy_z'].values  # canopy grid node heights [m], same grid both runs
hours = pd.DatetimeIndex(fixed_ds['date'].values).hour

TIME_MASKS = {
    'all-day': np.ones(len(hours), dtype=bool),
    'daytime (08-16)': (hours >= 8) & (hours < 16),
    'nighttime (22-06)': (hours >= 22) | (hours < 6),
}

print('Timesteps per period:', {k: int(v.sum()) for k, v in TIME_MASKS.items()})


In [ ]:
PROFILE_VARS = {
    'Tleaf': ('canopy_Tleaf', 'degC'),
    'H2O': ('canopy_h2o', 'mol mol-1'),
    'CO2': ('canopy_co2', 'ppm'),
}

fig, axes = plt.subplots(len(PROFILE_VARS), len(TIME_MASKS), figsize=(13, 11), sharey=True)
profile_changes = []

for i, (label, (varname, unit)) in enumerate(PROFILE_VARS.items()):
    base_var = baseline_ds[varname].values  # (date, canopy)
    fix_var = fixed_ds[varname].values
    for j, (period_label, mask) in enumerate(TIME_MASKS.items()):
        ax = axes[i, j]
        base_profile = np.nanmean(base_var[mask], axis=0)
        fix_profile = np.nanmean(fix_var[mask], axis=0)
        ax.plot(base_profile, z, color=COLOR_CRITICAL, label='before (case_ranskalankorpi)')
        ax.plot(fix_profile, z, color=COLOR_GOOD, label='now (fixed)')
        ax.set_title(f'{label}, {period_label}', fontsize=10)
        if j == 0:
            ax.set_ylabel('height [m]')
        if i == len(PROFILE_VARS) - 1:
            ax.set_xlabel(unit)
        for spine in ('top', 'right'):
            ax.spines[spine].set_visible(False)

        diff = fix_profile - base_profile
        profile_changes.append({
            'variable': label, 'period': period_label,
            'mean_change': np.nanmean(diff),
            'max_abs_change': np.nanmax(np.abs(diff)),
            'unit': unit,
        })

axes[0, 0].legend(fontsize=8, loc='best')
fig.suptitle('Canopy profiles: time-mean before vs now, by time of day')
plt.tight_layout()
plt.show()

profile_changes_df = pd.DataFrame(profile_changes)
profile_changes_df


In [ ]:
FLUX_VARS = {
    'NEE': ('canopy_NEE', 'umol m-2 s-1'),
    'GPP': ('canopy_GPP', 'umol m-2 s-1'),
    'Reco': ('canopy_Reco', 'umol m-2 s-1'),
    'H': ('canopy_SH', 'W m-2'),
    'LE': ('canopy_LE', 'W m-2'),
    'G': ('ffloor_ground_heat', 'W m-2'),
    'Rnet': ('canopy_Rnet', 'W m-2'),
}

dates = fixed_ds['date'].values
flux_changes = []

fig, axes = plt.subplots(len(FLUX_VARS), 1, figsize=(13, 2.2 * len(FLUX_VARS)), sharex=True)
for ax, (label, (varname, unit)) in zip(axes, FLUX_VARS.items()):
    base_s = pd.Series(baseline_ds[varname].values, index=dates)
    fix_s = pd.Series(fixed_ds[varname].values, index=dates)

    ax.plot(base_s.resample('1D').mean(), color=COLOR_CRITICAL, lw=1, label='before (case_ranskalankorpi)')
    ax.plot(fix_s.resample('1D').mean(), color=COLOR_GOOD, lw=1, label='now (fixed)')
    ax.set_ylabel(f'{label}\n[{unit}]', fontsize=9)
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)

    # stats computed on native 30-min resolution, not the daily means plotted above
    diff = fix_s.values - base_s.values
    flux_changes.append({
        'variable': label,
        'mean_change': np.nanmean(diff),
        'max_abs_change': np.nanmax(np.abs(diff)),
        'unit': unit,
    })

axes[0].legend(fontsize=8, loc='upper right')
axes[-1].set_xlabel('date')
fig.suptitle('Canopy-level fluxes: daily mean, before vs now (1.5.-30.9.2022)')
plt.tight_layout()
plt.show()

flux_changes_df = pd.DataFrame(flux_changes)
flux_changes_df


### Mean and max change, all plotted variables

`mean_change` and `max_abs_change` are `now (fixed) - before (case_ranskalankorpi)`.
Profile rows are computed on the time-mean profiles plotted above (mean/max taken over
height); flux rows are computed on the native 30-min series (mean/max taken over time,
not on the daily means plotted above).

In [ ]:
summary_df = pd.concat([
    profile_changes_df.assign(kind='profile', period=profile_changes_df['period']),
    flux_changes_df.assign(kind='flux', period='all-day'),
], ignore_index=True)[['kind', 'variable', 'period', 'unit', 'mean_change', 'max_abs_change']]

pd.set_option('display.float_format', lambda x: f'{x:.4g}')
summary_df


### Reading the flux changes

The magnitudes here are larger than the convergence-count section alone would suggest.
`Rnet`/`H`/`LE` mean changes of roughly -20 to -45 W m-2, and single-timestep swings up to
~260-350 W m-2, aren't explained only by the 1418/7297 timesteps the outer loop logged as
non-convergent -- the single biggest `Rnet`/`H`/`LE`/`NEE` differences found above land on
ordinary daytime timesteps with **no** outer-loop DEBUG message at all (e.g. 2022-06-13
10:00, 2022-05-30 10:00). That's consistent with the `interception`/`planttype` inner loops
silently hitting their own `itermax` on baseline (`case_ranskalankorpi` has no relaxation
there at all) without the error being large enough to trip the *outer* loop's convergence
check -- i.e. the outer loop can look "converged" while an inner loop's `Tleaf` solution was
still overshooting/oscillating internally. Net effect: the fix changes not just how many
timesteps need a fallback, but the leaf-temperature-dependent fluxes (`H`, `LE`, `Rnet`,
photosynthesis-linked `GPP`/`NEE`) on a meaningfully larger set of timesteps than the raw
non-convergence counts alone show.